# TF-IDF

In [ ]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.3 MB/s eta 0:00:00


In [ ]:
!pip install gensim

In [ ]:
# --- Import Library ---
import pandas as pd
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

1. Load Dataset

In [ ]:
def load_data(filepath="KompasBerita.csv"):
    df = pd.read_csv(filepath)

    if "judul" in df.columns and "isi" in df.columns:
        df["berita"] = df["judul"].astype(str) + " " + df["isi"].astype(str)
    elif "berita" in df.columns:
        df["berita"] = df["berita"].astype(str)
    else:
        df["berita"] = df.apply(lambda row: " ".join([str(x) for x in row]), axis=1)

    return df


In [ ]:
df = load_data("KompasBerita.csv")

2. Preprocessing

In [ ]:
stemmer = StemmerFactory().create_stemmer()
stop_factory = StopWordRemoverFactory()
stopwords = set(stop_factory.get_stop_words())

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(t) for t in tokens if t not in stopwords and len(t) > 2]
    return " ".join(tokens)

def preprocess_data(df):
    df["clean"] = df["berita"].apply(clean_text)
    return df[["berita", "clean"]]


In [ ]:
df_clean = preprocess_data(df)
display(df_clean.head())

,berita,clean
0,"Peristiwa Gas Air Mata Unisba, Mendikti Janjik...",peristiwa gas air mata unisba mendikti janji d...
1,Pendidikan Kewirausahaan yang Merdeka SETIAP17...,didik kewirausahaan merdeka agustus masyarakat...
2,Waspadai Kelelahan Mental akibat Kebanyakan Be...,waspada lelah mental akibat banyak berita nega...
3,"Berdarah Belanda Depok, Pesepak Bola Miliano J...",darah belanda depok sepak bola miliano jonatha...
4,Harapan dan Catatan soal Anggaran Pendidikan T...,harap catat soal anggar didik besar panjang se...


3. TF-IDF

In [ ]:
def compute_tfidf(df, max_features=20):
    vectorizer = TfidfVectorizer(max_features=max_features)
    tfidf_matrix = vectorizer.fit_transform(df["clean"])

    tfidf_df = pd.DataFrame(
        tfidf_matrix.toarray(),
        columns=vectorizer.get_feature_names_out()
    )
    tfidf_df.index = [f"Dokumen_{i+1}" for i in range(len(df))]

    print("Shape TF-IDF:", tfidf_matrix.shape)  # tampilkan shape
    return tfidf_df, tfidf_matrix

In [ ]:
tfidf_df, tfidf_matrix = compute_tfidf(df, max_features=20)
display(tfidf_df.head())

Shape TF-IDF: (5, 20)


,anggar,baca,belanda,berita,besar,depok,didik,guru,indonesia,informasi,jadi,jonathans,kampus,kata,kewirausahaan,miliano,negatif,perintah,sebut,tahun
Dokumen_1,0.000000,0.206966,0.000000,0.000000,0.000000,0.000000,0.193923,0.116808,0.000000,0.116808,0.163134,0.000000,0.872651,0.081567,0.000000,0.000000,0.000000,0.193923,0.244701,0.000000
Dokumen_2,0.000000,0.054944,0.000000,0.000000,0.038611,0.000000,0.270278,0.000000,0.487213,0.000000,0.162404,0.000000,0.038611,0.000000,0.807147,0.000000,0.000000,0.038611,0.000000,0.064962
Dokumen_3,0.000000,0.153070,0.000000,0.696009,0.071712,0.000000,0.000000,0.000000,0.030163,0.431950,0.090489,0.000000,0.000000,0.150815,0.000000,0.000000,0.481852,0.000000,0.180978,0.060326
Dokumen_4,0.000000,0.059238,0.372951,0.000000,0.000000,0.372951,0.000000,0.000000,0.087548,0.000000,0.052529,0.652665,0.000000,0.035019,0.000000,0.528348,0.000000,0.000000,0.017510,0.035019
Dokumen_5,0.732505,0.082128,0.000000,0.000000,0.259712,0.000000,0.490567,0.243345,0.097101,0.000000,0.000000,0.000000,0.028857,0.048551,0.000000,0.000000,0.000000,0.201998,0.121377,0.145652


4. Word Embedding (CBOW)

In [ ]:
def explore_word2vec(model, word="indonesia", similar_to="ekonomi"):
    # tampilkan shape embedding
    print("Shape Word Embedding (jumlah_kata, dimensi):", model.wv.vectors.shape)

    # Vektor kata → DataFrame
    if word in model.wv:
        vec = model.wv[word]
        vec_df = pd.DataFrame(vec, columns=[f"Nilai Vektor ({word})"])
    else:
        vec_df = pd.DataFrame([f"Kata '{word}' tidak ada di vocab"], columns=["Info"])

    # Kata mirip → DataFrame
    if similar_to in model.wv:
        sim_df = pd.DataFrame(model.wv.most_similar(similar_to, topn=5),
                              columns=["Kata", "Skor Similaritas"])
    else:
        sim_df = pd.DataFrame([f"Kata '{similar_to}' tidak ada di vocab"], columns=["Info"])

    return vec_df, sim_df

In [ ]:
model = train_word2vec(df, vector_size=100, window=5, min_count=1, sg=0)

In [ ]:
vec_df, sim_df = explore_word2vec(model, word="indonesia", similar_to="ekonomi")
display(vec_df.head(10))   # array embedding dalam bentuk tabel
display(sim_df)            # tabel kata mirip

Shape Word Embedding (jumlah_kata, dimensi): (627, 100)


,Nilai Vektor (indonesia)
0,-0.008551
1,0.004079
2,0.005326
3,0.005995
4,0.007674
5,-0.007148
6,0.001131
7,0.007118
8,-0.003458
9,-0.006291


,Kata,Skor Similaritas
0,ban,0.239816
1,insiden,0.238912
2,digital,0.232818
3,makan,0.223597
4,waspada,0.219937
